In [1]:
# Cell 1
%pip install -q -U transformers==4.46.3 peft==0.13.2 accelerate==0.34.2 datasets==2.21.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 71.1 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 80.3 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not inst

In [2]:
# Cell 2
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HOME"] = "/kaggle/working/hf_home_fix2"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

import json
import math
import random
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset, Features, Sequence, Value, concatenate_datasets
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, set_seed

SEED = 42

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = Path("/kaggle/working/qwen25_topical_chat_lora_fix2")
ADAPTER_DIR = OUTPUT_DIR / "adapter"

MAX_LENGTH = 320
MAX_HISTORY_TURNS = 8

TRAIN_FRACTION = 1
EVAL_FRACTION = 0.05

MAX_TRAIN_SAMPLES = 100000
MAX_EVAL_SAMPLES = 400

NUM_TRAIN_EPOCHS = 1
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRAD_ACCUM = 8

LEARNING_RATE = 1e-5
WARMUP_RATIO = 0.05

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

SYSTEM_PROMPT = "You are a helpful conversational assistant. Continue the conversation naturally and reply as the assistant."

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

assert torch.cuda.is_available(), "Enable GPU in Kaggle."

GPU_CAP_MAJOR, GPU_CAP_MINOR = torch.cuda.get_device_capability(0)
USE_AMP = GPU_CAP_MAJOR >= 7   # True on T4, False on P100

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.cuda.empty_cache()

print("GPU:", torch.cuda.get_device_name(0))
print("Capability:", (GPU_CAP_MAJOR, GPU_CAP_MINOR))
print("USE_AMP:", USE_AMP)

2026-05-09 21:51:26.100224: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778363486.481575      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778363486.595893      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778363487.503984      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778363487.504025      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778363487.504027      57 computation_placer.cc:177] computation placer alr

GPU: Tesla T4
Capability: (7, 5)
USE_AMP: True


In [3]:
# Cell 3
DATASET_HINT = Path("/kaggle/input/datasets/morichuzy/topical-chat-rag-data/topical-chat-rag-data")   # example: "/kaggle/input/topical-chat"

def find_conversations_dir(dataset_hint=None):
    if dataset_hint:
        p = Path(dataset_hint)
        if (p / "conversations" / "train.json").exists():
            return p / "conversations"
        if (p / "train.json").exists():
            return p

    matches = []
    for train_file in Path("/kaggle/input").rglob("train.json"):
        parent = train_file.parent
        if (parent / "valid_freq.json").exists() and (parent / "valid_rare.json").exists():
            matches.append(parent)

    if not matches:
        raise FileNotFoundError("Could not find train.json, valid_freq.json, valid_rare.json under /kaggle/input")

    return sorted(matches, key=lambda p: len(str(p)))[0]

CONV_DIR = find_conversations_dir(DATASET_HINT)
TRAIN_PATH = CONV_DIR / "train.json"
VALID_FREQ_PATH = CONV_DIR / "valid_freq.json"
VALID_RARE_PATH = CONV_DIR / "valid_rare.json"

print("CONV_DIR:", CONV_DIR)
print("TRAIN_PATH:", TRAIN_PATH)
print("VALID_FREQ_PATH:", VALID_FREQ_PATH)
print("VALID_RARE_PATH:", VALID_RARE_PATH)

CONV_DIR: /kaggle/input/datasets/morichuzy/topical-chat-rag-data/topical-chat-rag-data/conversations
TRAIN_PATH: /kaggle/input/datasets/morichuzy/topical-chat-rag-data/topical-chat-rag-data/conversations/train.json
VALID_FREQ_PATH: /kaggle/input/datasets/morichuzy/topical-chat-rag-data/topical-chat-rag-data/conversations/valid_freq.json
VALID_RARE_PATH: /kaggle/input/datasets/morichuzy/topical-chat-rag-data/topical-chat-rag-data/conversations/valid_rare.json


In [4]:
# Cell 4
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"
tokenizer.truncation_side = "left"

def normalize_text(text):
    return " ".join(str(text).strip().split())

def load_conversations(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict):
        return list(data.values())
    if isinstance(data, list):
        return data
    raise ValueError(f"Unsupported JSON format in {json_path}")

def render_chatml(history, add_generation_prompt=False):
    parts = [f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"]

    for msg in history:
        parts.append(f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>\n")

    if add_generation_prompt:
        parts.append("<|im_start|>assistant\n")

    return "".join(parts)

def tokenize_training_example(history, target_text):
    prompt_text = render_chatml(history, add_generation_prompt=True)
    target_text = normalize_text(target_text) + "<|im_end|>\n"

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    target_ids = tokenizer(target_text, add_special_tokens=False)["input_ids"]

    if len(target_ids) < 2:
        return None

    total_len = len(prompt_ids) + len(target_ids)

    if total_len > MAX_LENGTH:
        overflow = total_len - MAX_LENGTH

        # skip example if truncation would cut into the target
        if overflow >= len(prompt_ids):
            return None

        prompt_ids = prompt_ids[overflow:]

    input_ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids
    attention_mask = [1] * len(input_ids)

    target_tokens = len(target_ids)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "length": len(input_ids),
        "target_tokens": target_tokens,
    }

def iter_tokenized_examples(json_paths, fraction=None, seed=SEED):
    rng = random.Random(seed)

    for json_path in json_paths:
        conversations = load_conversations(json_path)

        if fraction is not None and 0 < fraction < 1.0:
            rng.shuffle(conversations)
            keep_n = max(1, int(len(conversations) * fraction))
            conversations = conversations[:keep_n]

        for conv in conversations:
            history = []

            for turn in conv.get("content", []):
                text = normalize_text(turn.get("message", ""))
                if not text:
                    continue

                role = "assistant" if turn.get("agent") == "agent_2" else "user"

                if role == "assistant":
                    example = tokenize_training_example(history, text)
                    if example is not None:
                        yield example

                history.append({"role": role, "content": text})
                history = history[-MAX_HISTORY_TURNS:]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
# Cell 5
FEATURES = Features(
    {
        "input_ids": Sequence(Value("int32")),
        "attention_mask": Sequence(Value("int8")),
        "labels": Sequence(Value("int32")),
        "length": Value("int32"),
        "target_tokens": Value("int32"),
    }
)

train_dataset = Dataset.from_generator(
    iter_tokenized_examples,
    gen_kwargs={
        "json_paths": [str(TRAIN_PATH)],
        "fraction": TRAIN_FRACTION,
        "seed": SEED,
    },
    features=FEATURES,
    cache_dir=os.environ["HF_HOME"],
)

eval_freq_dataset = Dataset.from_generator(
    iter_tokenized_examples,
    gen_kwargs={
        "json_paths": [str(VALID_FREQ_PATH)],
        "fraction": EVAL_FRACTION,
        "seed": SEED + 1,
    },
    features=FEATURES,
    cache_dir=os.environ["HF_HOME"],
)

eval_rare_dataset = Dataset.from_generator(
    iter_tokenized_examples,
    gen_kwargs={
        "json_paths": [str(VALID_RARE_PATH)],
        "fraction": EVAL_FRACTION,
        "seed": SEED + 2,
    },
    features=FEATURES,
    cache_dir=os.environ["HF_HOME"],
)

eval_dataset = concatenate_datasets([eval_freq_dataset, eval_rare_dataset])

train_dataset = train_dataset.shuffle(seed=SEED)
eval_dataset = eval_dataset.shuffle(seed=SEED)

if MAX_TRAIN_SAMPLES is not None:
    train_dataset = train_dataset.select(range(min(MAX_TRAIN_SAMPLES, len(train_dataset))))

if MAX_EVAL_SAMPLES is not None:
    eval_dataset = eval_dataset.select(range(min(MAX_EVAL_SAMPLES, len(eval_dataset))))

print("train samples:", len(train_dataset))
print("eval samples:", len(eval_dataset))

for i in range(min(32, len(train_dataset))):
    assert train_dataset[i]["target_tokens"] > 0, f"Bad train sample {i}"

for i in range(min(32, len(eval_dataset))):
    assert eval_dataset[i]["target_tokens"] > 0, f"Bad eval sample {i}"

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

train samples: 91174
eval samples: 400


In [6]:
# Cell 6
sample = train_dataset[0]

target_ids = [tok for tok, lbl in zip(sample["input_ids"], sample["labels"]) if lbl != -100]
prompt_ids = [tok for tok, lbl in zip(sample["input_ids"], sample["labels"]) if lbl == -100]

print("sample length:", sample["length"])
print("target token count:", sample["target_tokens"])

print("\nPROMPT TAIL:\n")
print(tokenizer.decode(prompt_ids[-200:], skip_special_tokens=False))

print("\nTARGET:\n")
print(tokenizer.decode(target_ids, skip_special_tokens=False))

sample length: 233
target token count: 12

PROMPT TAIL:

<|im_end|>
<|im_start|>assistant
At least it still requires pratice, development and refinement web or printed.<|im_end|>
<|im_start|>user
yes, it sure does. it seems like an easy skill but careful reading requires practice.<|im_end|>
<|im_start|>assistant
yes in addition to creativity and critical analysis<|im_end|>
<|im_start|>user
yes, i think that the mind gets a gymnastic workout from reading. literacy is a fundamental human right.<|im_end|>
<|im_start|>assistant
did you know the speed reading record is 4700 words per minute?<|im_end|>
<|im_start|>user
Can anyone remember what they read at that speed? i cant. what is the point? I would rather enjoy what i read.<|im_end|>
<|im_start|>assistant
Her name is anne jones. They have to test her for comprehension otherwise it would not count I would think?<|im_end|>
<|im_start|>user
yeah, i agree. how many details can she remember? i think it is useful to read faster, but not that f

In [7]:
# Cell 7
class CausalLMCollator:
    def __init__(self, tokenizer):
        self.pad_token_id = tokenizer.pad_token_id

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)

        input_ids = []
        attention_mask = []
        labels = []

        for f in features:
            pad_len = max_len - len(f["input_ids"])
            input_ids.append(f["input_ids"] + [self.pad_token_id] * pad_len)
            attention_mask.append(f["attention_mask"] + [0] * pad_len)
            labels.append(f["labels"] + [-100] * pad_len)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

torch.cuda.empty_cache()

# CRITICAL: load base model in float32, not float16
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)

model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id



lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.enable_input_require_grads()
model.print_trainable_parameters()

model = model.cuda()
data_collator = CausalLMCollator(tokenizer)

sanity_batch = data_collator([train_dataset[0]])
sanity_batch = {k: v.cuda() for k, v in sanity_batch.items()}

model.eval()
with torch.no_grad():
    sanity_loss = model(**sanity_batch).loss

print("sanity loss:", float(sanity_loss))
assert torch.isfinite(sanity_loss), "Sanity loss still non-finite. Restart notebook and ensure model loads in float32."

model.train()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410
sanity loss: 4.026722431182861


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2SdpaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linea

In [8]:
# Cell 7.5
@torch.no_grad()
def scan_losses(ds, name, max_items=128):
    model.eval()
    bad = []

    for i in range(min(max_items, len(ds))):
        batch = data_collator([ds[i]])
        batch = {k: v.cuda() for k, v in batch.items()}
        loss = model(**batch).loss

        if not torch.isfinite(loss):
            bad.append(i)
            print(f"[{name}] bad sample index:", i)
            print("length:", ds[i]["length"])
            print("target_tokens:", ds[i]["target_tokens"])

            target_ids = [tok for tok, lbl in zip(ds[i]["input_ids"], ds[i]["labels"]) if lbl != -100]
            print("\nTARGET TEXT:\n")
            print(tokenizer.decode(target_ids[:300], skip_special_tokens=False))
            break

    model.train()

    if not bad:
        print(f"[{name}] first {min(max_items, len(ds))} samples are finite.")

scan_losses(train_dataset, "train", max_items=128)
scan_losses(eval_dataset, "eval", max_items=128)

[train] first 128 samples are finite.
[eval] first 128 samples are finite.


In [ ]:
# Cell 8
class StableTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        loss = outputs["loss"] if isinstance(outputs, dict) else outputs.loss

        if not torch.isfinite(loss):
            raise RuntimeError("NaN/Inf loss detected.")

        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    overwrite_output_dir=True,
    num_train_epochs=2,
    max_steps=-1,                   # quick debug run
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    max_grad_norm=0.3,
    fp16=False,
    bf16=False,
    optim="adamw_torch",
    logging_steps=5,
    logging_first_step=True,
    eval_strategy="no",             # skip slow eval during debug
    save_strategy="no",             # skip slow checkpointing during debug
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    dataloader_pin_memory=True,
    gradient_checkpointing=True,
    group_by_length=False,
    prediction_loss_only=True,
    seed=SEED,
)

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

try:
    trainer = StableTrainer(processing_class=tokenizer, **trainer_kwargs)
except TypeError:
    trainer = StableTrainer(tokenizer=tokenizer, **trainer_kwargs)

trainer.model_accepts_loss_kwargs = False

train_result = trainer.train()
print(train_result.metrics)

In [ ]:
# Cell 8.5 - Evaluation + Perplexity
import math
import torch

# Use a small eval subset first to avoid Kaggle hanging/slowing down.
# Increase this later, or set EVAL_SAMPLES = None for full eval.
EVAL_SAMPLES = 100

if EVAL_SAMPLES is None:
    eval_for_metrics = eval_dataset
else:
    eval_for_metrics = eval_dataset.select(range(min(EVAL_SAMPLES, len(eval_dataset))))

torch.cuda.empty_cache()

trainer.model.eval()

eval_metrics = trainer.evaluate(
    eval_dataset=eval_for_metrics,
    metric_key_prefix="eval",
)

eval_loss = eval_metrics["eval_loss"]

if math.isfinite(eval_loss) and eval_loss < 20:
    perplexity = math.exp(eval_loss)
else:
    perplexity = float("inf")

eval_metrics["perplexity"] = perplexity

print("Evaluation metrics:")
print(eval_metrics)

print(f"\nEval loss: {eval_loss:.4f}")
print(f"Perplexity: {perplexity:.4f}")

In [ ]:
# quick eval
small_eval = eval_dataset.select(range(min(50, len(eval_dataset))))
trainer.eval_dataset = small_eval
eval_metrics = trainer.evaluate()
print(eval_metrics)

In [ ]:
# Cell 9
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print("saved adapter:", ADAPTER_DIR)

In [ ]:
# Cell 10
del trainer
del model
torch.cuda.empty_cache()

infer_tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR), use_fast=True)
if infer_tokenizer.pad_token_id is None:
    infer_tokenizer.pad_token = infer_tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True,
    attn_implementation="eager",
).cuda()

chat_model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
chat_model.eval()

In [ ]:
# Cell 11
@torch.inference_mode()
def chat_once(user_text, history=None, max_new_tokens=96):
    history = [] if history is None else history

    history_for_prompt = history + [{"role": "user", "content": normalize_text(user_text)}]
    prompt_text = render_chatml(history_for_prompt, add_generation_prompt=True)

    input_ids = infer_tokenizer(prompt_text, add_special_tokens=False, return_tensors="pt")["input_ids"]
    attention_mask = torch.ones_like(input_ids)

    input_ids = input_ids.cuda()
    attention_mask = attention_mask.cuda()

    output = chat_model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.1,
        use_cache=True,
        pad_token_id=infer_tokenizer.pad_token_id,
        eos_token_id=infer_tokenizer.eos_token_id,
    )

    new_tokens = output[0, input_ids.shape[1]:]
    reply = infer_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    history = history + [
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": reply},
    ]
    history = history[-MAX_HISTORY_TURNS:]

    return reply, history

history = []
reply, history = chat_once("What is football?", history)
print(reply)

In [ ]:
# Cell 12
history = []

while True:
    user_text = input("User: ").strip()

    if not user_text:
        continue
    if user_text.lower() in {"exit", "quit"}:
        break
    if user_text.lower() == "/reset":
        history = []
        print("History cleared.\n")
        continue

    reply, history = chat_once(user_text, history)
    print(f"Assistant: {reply}\n")